In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from datasets import load_dataset
import torch
import wandb

wandb.init(project='mini-LLM', name='pair_generation')

sft_path = "./checkpoints/sft/sft-seed42/final"
rm_path = "./checkpoints/rm/final"

model_sft = AutoModelForCausalLM.from_pretrained(sft_path, dtype=torch.bfloat16, device_map="auto")
model_rm = AutoModelForSequenceClassification.from_pretrained(rm_path, num_labels=1, dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(sft_path)

tokenizer.padding_side = "left"
tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
model_sft.resize_token_embeddings(len(tokenizer))

/home/artmak/Desktop/Education/mini-LLM/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/artmak/.netrc.
wandb: Currently logged in as: artmak (artmak-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 5879.89it/s]


Embedding(151666, 896, padding_idx=151665)

In [2]:
dataset_size = 3000
offside = 100_000
random_state=42

train_dataset = load_dataset("HuggingFaceTB/smoltalk2",
    "SFT",
    split="OpenHermes_2.5_no_think"
    ).select(range(offside, offside + dataset_size))

print(train_dataset[0])

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
eos_ids = [tokenizer.eos_token_id, im_end_id]

{'messages': [{'content': 'Process:  - An owl leaves his nest - The owl flys out into the night - The owl looks down at the ground - The owls keen eyesight allows him to see in the dark - The owls sensitive hearing allows him to locate hidden prey - The owl swoops down - The owl catches a mouse in his talons.  Perturbation hypothesis: suppose The owl is feeding less young happens, how will it affect The owl catches more mice.  Does the supposed perturbation have an effect (direct or indirect) on the process?', 'role': 'user'}, {'content': 'Step 1: Analyzing the process:\n\nThe process involves an owl, which performs the following actions:\n  a. Leaves its nest\n  b. Flies out into the night\n  c. Looks down at the ground\n  d. Uses its keen eyesight to see in the dark\n  e. Uses its sensitive hearing to locate hidden prey\n  f. Swoops down\n  g. Catches a mouse in its talons.\n\nStep 2: Understanding the perturbation hypothesis:\n\nThe perturbation hypothesis states: "Suppose the owl i

In [3]:
import torch

def generate_responses(prompt_messages, n=8):
    messages = [{"role": "user", "content": prompt_messages[0]["content"]}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model_sft.device)
    with torch.no_grad():
            outputs = model_sft.generate(
                **inputs,
                num_return_sequences=n,
                max_new_tokens=256,
                temperature=0.9,
                top_p=0.95,
                do_sample=True,
                eos_token_id=eos_ids,
                pad_token_id=tokenizer.pad_token_id,
            )
    responses = []
    for out in outputs:
          text = tokenizer.decode(out[inputs['input_ids'].shape[1]:], skip_special_tokens=True)
          responses.append(text)
    return messages, responses

def score_messages(messages, responses):
      conversations = messages + [{"role":"assistant", "content":responses}]
      text = tokenizer.apply_chat_template(conversations, tokenize=False)
      inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(model_rm.device)

      with torch.no_grad():
            reward = model_rm(**inputs).logits.squeeze().item()
      return reward

msgs, resps = generate_responses(train_dataset[0]["messages"], n=3)
for i, r in enumerate(resps):
      score = score_messages(msgs, r)
      print(f"Response {i}: score - {score:.4f} | {r[:100]}")

Response 0: score - 1.1172 | The supposed perturbation, "the owl is feeding less young," does not have a direct or indirect effec
Response 1: score - -0.4668 | To determine the effect of feeding less young on the process, let's analyze it step by step.

1. Und
Response 2: score - -0.1445 | Yes, the supposed perturbation of the owl feeding less young would have a direct effect on the proce


In [4]:
num_responses = 8
min_reward_std = 0.151

In [6]:
import numpy as np


stds = []
for idx in range(100):
    msgs, resps = generate_responses(train_dataset[idx]["messages"], n=num_responses)
    rewards = [score_messages(msgs, r) for r in resps]
    stds.append(np.std(rewards))

print(f"mean std: {np.mean(stds):.3f}")
print(f"median std: {np.median(stds):.3f}")
print(f"10th percentile: {np.percentile(stds, 10):.3f}")

mean std: 0.442
median std: 0.354
10th percentile: 0.136


In [7]:
import time

pairs = []
discarded = 0
gpu_start = time.time()

for i in range(len(train_dataset)):
    msgs, resps = generate_responses(train_dataset[i]["messages"], n=num_responses)
    rewards = [score_messages(msgs, r) for r in resps]

    std = np.std(rewards)
    if std < min_reward_std:
        discarded += 1
        continue

    best_idx = int(np.argmax(rewards))
    worst_idx = int(np.argmin(rewards))

    pairs.append({
        "prompt": msgs[0]["content"],
        "chosen": resps[best_idx],
        "rejected": resps[worst_idx],
        "reward_chosen": rewards[best_idx],
        "reward_rejected": rewards[worst_idx],
        "reward_std": float(std),
    })

    if (i + 1) % 100 == 0:
        elapsed = time.time() - gpu_start
        print(f"[{i+1}/{len(train_dataset)}] pairs={len(pairs)} discarded={discarded} time={elapsed:.0f}s")

gpu_hours = (time.time() - gpu_start) / 3600
print(f"\nDone. Pairs: {len(pairs)}, Discarded: {discarded} ({discarded/(len(pairs)+discarded)*100:.1f}%)")
print(f"GPU time: {gpu_hours:.2f} hours")

[100/3000] pairs=90 discarded=10 time=132s
[200/3000] pairs=175 discarded=25 time=257s
[300/3000] pairs=263 discarded=37 time=384s
[400/3000] pairs=354 discarded=46 time=503s
[500/3000] pairs=445 discarded=55 time=631s
[600/3000] pairs=540 discarded=60 time=761s
[700/3000] pairs=624 discarded=76 time=879s
[800/3000] pairs=715 discarded=85 time=1006s
[900/3000] pairs=802 discarded=98 time=1131s
[1000/3000] pairs=893 discarded=107 time=1251s
[1100/3000] pairs=977 discarded=123 time=1369s
[1200/3000] pairs=1065 discarded=135 time=1501s
[1300/3000] pairs=1152 discarded=148 time=1632s
[1400/3000] pairs=1242 discarded=158 time=1756s
[1600/3000] pairs=1424 discarded=176 time=2001s
[1700/3000] pairs=1509 discarded=191 time=2123s
[1800/3000] pairs=1597 discarded=203 time=2248s
[1900/3000] pairs=1687 discarded=213 time=2370s
[2000/3000] pairs=1778 discarded=222 time=2490s
[2100/3000] pairs=1865 discarded=235 time=2610s
[2200/3000] pairs=1951 discarded=249 time=2734s
[2400/3000] pairs=2117 discar

In [8]:
import json

output = {
    "pairs": pairs,
    "metadata": {
        "gpu_hours": gpu_hours,
        "total_prompts": len(train_dataset),
        "valid_pairs": len(pairs),
        "discarded": discarded,
        "discard_rate": discarded / (len(pairs) + discarded),
        "min_reward_std": min_reward_std,
        "num_responses_per_prompt": num_responses,
    }
}

with open("dpo_pairs.json", "w") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Saved {len(pairs)} pairs to dpo_pairs.json")
print(f"GPU hours (DPO data cost): {gpu_hours:.2f}")

Saved 2644 pairs to dpo_pairs.json
GPU hours (DPO data cost): 1.03
